# 06B — Comparative Semantic Probing: the dual space on weights and embeddings

The use of feature-space in arrowspace makes possible the leveraging of a dual space: the geometric space of the vectors and the semantic space of the features.


> **Objective** -- This notebooks computes and visualises the activations in a BERT model for the model's inputs
> values and the trasformer pass forward values.
>
> **Methodology** — Using `all-MiniLM-L6-v2`, we extract layer-wise activation patterns from a limited-vocabulary text corpus and compare the flow of energy in the activations for the inputs and the full-pass values. This is done by treating the final layer in a dual aspect of geometric space ($S$) and semantic space ($S^\top$).

---

### Unique angle — weight-space probing

Unlike prior notebooks that probe *embedding space*, this notebook probes  
**where the attention and FFN weight matrices themselves place semantic fields**.  
The key insight: weight matrices of a frozen LM are a compressed spectral encoding  
of the pre-training corpus topology. By treating each row of `W_q / W_k / W_v / W_o`  
as a latent "neuron direction" and projecting text embeddings onto them layer by layer,  
we obtain *layer-wise activation patterns* that carry mechanistic-interpretability  
semantics — distinct from the final `[CLS]` embedding.


---
## 0 · Imports and constants

In [9]:
# ── stdlib / data ─────────────────────────────────────────────────────────
import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# ── ML / embedding ─────────────────────────────────────────────────────────
import torch
from sentence_transformers import SentenceTransformer

# ── ArrowSpace ─────────────────────────────────────────────────────────────
from arrowspace import ArrowSpaceBuilder              # pip install arrowspace

# ── Analysis / viz ─────────────────────────────────────────────────────────
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity
from sklearn.metrics import pairwise_distances
from scipy.stats import pearsonr
from scipy.spatial.distance import cdist
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ── Hyper-parameters ───────────────────────────────────────────────────────
ARROW_MAG   = 1.12   # magnification applied to ArrowSpace (better for dimensions clustering)
N_WORDS     = 200    # vocabulary size for the probing corpus
KNN_K       = 12     # k-NN for ArrowSpace graph wiring
ALPHA_STEPS = 11     # number of α values in [0, 1] sweeps
TOP_K_PCT   = 0.15   # fraction of items treated as "basin minima"

OUTPUT_DIR = Path("output__06")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Imports OK. Output →", OUTPUT_DIR)


Imports OK. Output → output__06


### Experiment: static word embedding matrix and full pass

> PROBE A: `E_tok` is the static word embedding matrix — a lookup table initialised before any training signal propagates through attention or FFN layers. Each row is a 384-dim vector assigned to a token at the very first stage of the forward pass, before any contextualisation occurs. Using `E_tok[token_id]` gives you the pre-attention representation of a word in complete isolation.

> PROBE B: `model.encode(["cat"])` runs the full 6-layer transformer: the token embedding is retrieved, then passed through all attention + FFN layers, then mean-pooled. The result encodes not just "what token is this" but "how this token relates to all other tokens it co-occurs with in pre-training" — the entire contextual geometry learned by the model.

| Aspect | E_tok (raw lookup) | model.encode() (full pass) |
| :-- | :-- | :-- |
| **What it represents** | Pre-attention token direction | Contextualised semantic embedding |
| **Interaction with W_q/W_k etc.** | Projection of the *input* before any layer has seen it | Projection of the *output* after all layers have processed it |
| **Semantic field separation** | Weaker — very similar words share overlapping token vectors | Stronger — contextual geometry better separates fields |
| **What §3 actually measures** | How W matrices *receive* raw token signals | How W matrices *respond to* already-contextualised meanings |
| **Single-token claim validity** | ✅ True — one `E_tok` row, no subword averaging | ❌ False — even a single word gets full attention over its own BOS/EOS tokens |
| **Mechanistic interpretability value** | More tractable — directly ties to circuit analysis | More downstream — measures emergent representation not weight structure |


Using `E_tok`: "where do attention and FFN weight matrices place semantic fields" — using E_tok would be more faithful to the mechanistic-interpretability claim. You'd be asking: given a raw token direction, how do the weights transform it? That's a direct circuit-level question.

Using model.encode(), you're asking: given the model's final opinion of a word, how do the weights respond? The weights have already shaped those embeddings, so the projection in §3 is partially circular — the W_q at layer 3 helped create the X_pass you're projecting through it. This introduces a mild self-consistency bias that inflates activation energies for fields the model represents strongly, independent of what the raw weight geometry does.

---
## 1 · Load model and extract weight matrices

We load all-MiniLM-L6-v2 and extract the six layers of

Q / K / V / O / FFN-up / FFN-down
weight matrices.Each matrix is stored in a dict keyed by (layer_idx, role).

For FFN we distinguish:

W_ffn1 (**primal**): the up-projection from the 384‑dim token space into the 1536‑dim FFN hidden space.

W_ffn2 (**readout**): the down-projection from the 1536‑dim FFN hidden space back to the 384‑dim residual stream.

In §3, we will probe `W_ffn1` as a primal “write into FFN” operator and `W_ffn2` via its transpose as a dual/readout operator, analogous to ArrowSpace’s feature‑spectral (transposed) view.


In [10]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

bert = model[0].auto_model          # transformers.BertModel
layers = bert.encoder.layer         # ModuleList of 6 BertLayer

# Collect weight matrices per layer
WEIGHT_ROLES = ["W_q", "W_k", "W_v", "W_o", "W_ffn1", "W_ffn2"]
weights = {}

for i, layer in enumerate(layers):
    attn = layer.attention.self
    weights[(i, "W_q")]    = attn.query.weight.detach().numpy()          # (384, 384)
    weights[(i, "W_k")]    = attn.key.weight.detach().numpy()            # (384, 384)
    weights[(i, "W_v")]    = attn.value.weight.detach().numpy()          # (384, 384)
    weights[(i, "W_o")]    = layer.attention.output.dense.weight \
                                 .detach().numpy()                        # (384, 384)
    weights[(i, "W_ffn1")] = layer.intermediate.dense.weight \
                                 .detach().numpy()                        # (1536, 384)
    weights[(i, "W_ffn2")] = layer.output.dense.weight \
                                 .detach().numpy()                        # (384, 1536)

# Also keep the token embedding matrix E ∈ ℝ^{V × 384}
E_tok = bert.embeddings.word_embeddings.weight.detach().numpy()          # (30522, 384)

print(f"Extracted {len(weights)} weight matrices across {len(layers)} layers.")
for (i, role), W in list(weights.items())[:6]:
    print(f"  Layer {i} | {role:7s} → shape {W.shape}")


Extracted 36 weight matrices across 6 layers.
  Layer 0 | W_q     → shape (384, 384)
  Layer 0 | W_k     → shape (384, 384)
  Layer 0 | W_v     → shape (384, 384)
  Layer 0 | W_o     → shape (384, 384)
  Layer 0 | W_ffn1  → shape (1536, 384)
  Layer 0 | W_ffn2  → shape (384, 1536)


---
## 2 · Build a limited-vocabulary probing corpus

We select `N_WORDS = 200` semantically diverse single-token words
drawn from 10 semantic fields (20 words each).
Ground-truth labels come from those 10 fields.

> **Why single-token words?**
> Constraining the corpus to single-token words ensures each item maps to
> *exactly one* row in `E_tok` — eliminating subword averaging artefacts.
> However, this does **not** mean the two embedding strategies are equivalent:
>
> | Strategy | What it encodes | Question asked of the weights |
> |---|---|---|
> | `X_etok` — raw `E_tok[token_id]` | Pre-attention token direction; no contextualisation | *"How do the weight matrices transform a raw token signal?"* — a direct circuit-level question |
> | `X_pass` — `model.encode(word)` | Full 6-layer contextualised representation | *"How do weights respond to the model's own final opinion of a word?"* — partially circular: W_q at layer 3 helped shape the embedding being projected through it |
>
> **We run both** and compare them in §3.1. The gap between their activation
> energy profiles directly reveals **how much each layer's weights reshape
> the raw token geometry** — arguably the most interesting diagnostic in this notebook.
>
> Concretely: if `X_etok` and `X_pass` produce identical field-separation
> patterns, the attention layers add no new geometric structure to the
> token directions. Divergence indicates the layers do non-trivial re-encoding.

In [18]:
SEMANTIC_FIELDS = {
    "FOOD": [
        "bread", "rice", "soup", "cake", "pizza", "pasta", "salad",
        "curry", "cheese", "butter", "cream", "jam", "honey",
        "chocolate", "coffee", "tea", "wine", "beer", "milk", "sugar"
    ],
    "SCIENCE": [
        "atom", "electron", "proton", "neutron", "photon", "atom",
        "force", "energy", "mass", "gravity", "entropy", "plasma",
        "laser", "magnet", "circuit", "gene", "cell", "virus",
        "enzyme", "protein"
    ],
    "TOOL": [
        "hammer", "saw", "drill", "drill", "screw", "nail", "bolt",
        "knife", "blade", "hook", "forge", "wheel", "axle",
        "lever", "axle", "gear", "spring", "joint", "vice", "hook"
    ],
    "COLOUR": [
        "red", "blue", "green", "yellow", "purple", "orange", "pink",
        "brown", "black", "white", "grey", "grey", "violet", "gold",
        "silver", "beige", "azure", "indigo", "violet", "crimson"
    ],
}

FIELD_COLOURS = {
    "ANIMAL":   "#e6194b", "FOOD":     "#f58231", "EMOTION":  "#ffe119",
    "SCIENCE":  "#3cb44b", "PLACE":    "#42d4f4", "TOOL":     "#4363d8",
    "MUSIC":    "#911eb4", "COLOUR":   "#f032e6", "ACTION":   "#a9a9a9",
    "ABSTRACT": "#9A6324",
}


words, labels = [], []
for field, wlist in SEMANTIC_FIELDS.items():
    for w in wlist[:20]:
        words.append(w)
        labels.append(field)

labels = np.array(labels)
print(f"Corpus: {len(words)} words across {len(SEMANTIC_FIELDS)} semantic fields.")

# ── Probe A: raw token embedding lookup (E_tok) ────────────────────────────
# E_tok[token_id] retrieves the pre-attention embedding vector — the signal the
# model receives BEFORE any attention or FFN layer processes it.
# Question asked: "How do the weight matrices transform a raw token direction?"
# This is the more faithful mechanistic-interpretability probe: it tests the
# weight geometry independent of what the model has already learned to produce.
# Single-token constraint guarantees a 1-to-1 mapping: one word → one E_tok row.
tokenizer = model.tokenizer
token_ids = []
multi_token_words = []
for w in words:
    ids = tokenizer.encode(w, add_special_tokens=False)
    if len(ids) != 1:
        multi_token_words.append((w, ids))
    token_ids.append(ids[0] if ids else tokenizer.unk_token_id)

if multi_token_words:
    # check if there are multi-token workds
    raise AssertionError(
        f"Single-token constraint violated for {len(multi_token_words)} word(s):\n"
        + "\n".join(f"  '{w}' → {ids}" for w, ids in multi_token_words)
    )

token_ids = np.array(token_ids)

X_etok_raw  = E_tok[token_ids]                        # (200, 384), raw lookup
X_etok      = normalize(X_etok_raw, norm="l2")        # for vanilla branches
X_etok_arrow = X_etok * ARROW_MAG                    # for ArrowSpace branch

# ── Probe B: full transformer pass (model.encode) ──────────────────────────
# model.encode(word) runs all 6 attention + FFN layers, then mean-pools.
# The result encodes how the model contextualises the word given its pre-training.
# NOTE: This creates a mild self-consistency bias when projecting through weight
# matrices in §3 — e.g. W_q at layer 3 partially shaped this embedding already,
# so the activation energy measures "how well weights recognise their own output"
# rather than purely "how weights transform an external signal".
X_pass_raw  = model.encode(words, batch_size=64, show_progress_bar=False,
                      convert_to_numpy=True)
X_pass  = normalize(X_pass_raw,  norm="l2")   # for vanilla branches
X_arrow = X_pass * ARROW_MAG            # for ArrowSpace branch


# ── Sanity check: per-item cosine similarity between strategies ───────────────
# High similarity → transformer adds little new directional information for that word.
# Low similarity → substantial re-encoding across the 6 layers.
cos_sim = np.einsum("ij,ij->i", X_pass, X_etok)      # dot of two L2-normed vecs = cosine
print(f"\nProbe A vs B — cosine similarity (per item):")
print(f"  mean={cos_sim.mean():.4f}  std={cos_sim.std():.4f}  "
      f"min={cos_sim.min():.4f}  max={cos_sim.max():.4f}")

# Per-field mean cosine (fields with low similarity are most re-encoded by layers)
df_cos = pd.DataFrame({"word": words, "field": labels, "cos_sim": cos_sim})
print("\nMean cosine similarity (model.encode vs E_tok) per semantic field:")
print(df_cos.groupby("field")["cos_sim"].mean().sort_values().to_string())

print(f"\nX_pass   shape: {X_pass.shape}  (L2-normalised, full transformer pass)")
print(f"X_etok   shape: {X_etok.shape}  (L2-normalised, raw E_tok lookup)")
print(f"X_arrow  shape: {X_arrow.shape} (magnified × {ARROW_MAG}, full pass)")
print(f"X_etok_arrow shape: {X_etok_arrow.shape} (magnified × {ARROW_MAG}, E_tok)")


Corpus: 80 words across 4 semantic fields.

Probe A vs B — cosine similarity (per item):
  mean=0.3020  std=0.0626  min=0.1376  max=0.4221

Mean cosine similarity (model.encode vs E_tok) per semantic field:
field
COLOUR     0.222658
FOOD       0.325110
SCIENCE    0.328963
TOOL       0.331138

X_pass   shape: (80, 384)  (L2-normalised, full transformer pass)
X_etok   shape: (80, 384)  (L2-normalised, raw E_tok lookup)
X_arrow  shape: (80, 384) (magnified × 1.12, full pass)
X_etok_arrow shape: (80, 384) (magnified × 1.12, E_tok)


---
### Theoretical bridge — activation energy as a Rayleigh quotient analogue

The `activation_energy` function defined in §0 computes:

$$
E(W, x) = \frac{\|W\, x\|_2}{\|W\|_F + \varepsilon}
$$

This is a **Frobenius-normalised projection norm** — a scalar that measures how strongly
the weight matrix $W$ amplifies the direction of an input token vector $x$.

This quantity is a linear analogue of the **Rayleigh quotient** that ArrowSpace uses
internally to define the $\lambda$-score:

$$
R(x) = \frac{x^\top L\, x}{x^\top x}
\quad \xrightarrow{\text{ArrowSpace}} \quad
\lambda_w(x) = w \cdot R_{\text{geom}}(x) + (1-w) \cdot R_{\text{spec}}(x)
$$

where $L$ is the normalised graph Laplacian built from the k-NN feature graph (see
[`notebooks/README.md §2`](../README.md)).

**The analogy holds at two levels:**

| ArrowSpace ($\lambda$) | Weight-space probe ($E$) |
|:--|:--|
| $L = \Phi\,\Lambda\,\Phi^\top$ — Laplacian of the *data* graph | $W$ — a single transformer weight matrix (Q/K/V/O/FFN) |
| $R(x) = x^\top L\, x / \|x\|^2$ — energy of $x$ on the data manifold | $E(W,x) = \|Wx\|_2 / \|W\|_F$ — energy of $x$ in the weight subspace |
| Low $\lambda$ → $x$ lies in a smooth, dense semantic basin | Low $E$ → $x$ is weakly activated by that weight matrix |
| High $\lambda$ → $x$ is at a spectral boundary or anomaly | High $E$ → $x$ strongly excites the weight direction (salient circuit) |

**Key difference:** $R(x)$ is built from the *dataset topology* (how items relate to each
other via the k-NN graph). $E(W, x)$ is built from *model topology* (how a frozen weight
matrix responds to a token direction). The gap between Probe A  (`E_tok`) and
Probe B (`model.encode`) — quantified in §3 — directly reveals **how much the transformer's
learned weight geometry diverges from the raw token-embedding geometry**: a divergence
that ArrowSpace's spectral component $R_{\text{spec}}$ is designed to capture at inference
time.

---
## 3 · Semantic Subspace Matrix Diagram

These cells explicitly answer the question:

> **"Which subspaces of the model's latent space encode which semantic fields?"**

For every combination of **(layer × weight-role)** we compute how strongly each  
semantic field *dominates* that subspace.  Domination is measured as:

$$S_{(i,r,f)} = \frac{\bar{A}_{(i,r,f)} - \bar{A}_{(i,r,\neg f)}}{\bar{A}_{(i,r,f)} + \bar{A}_{(i,r,\neg f)} + \varepsilon}$$

where $\bar{A}_{(i,r,f)}$ is the mean activation energy of field $f$'s words  
in subspace $(i, r)$.  This normalised contrast score $\in [-1, +1]$  
identifies subspaces where a field's activation is unusually high relative  
to all other fields.

> **Primal vs dual FFN probes**
>
> For attention and FFN we separate:
> - **Primal roles**: `W_q`, `W_k`, `W_v`, `W_o`, `W_ffn1`
>   These take a 384‑dim token direction $x$ and measure a standard projection norm $\|W x\|_2 / \|W\|_F$. They answer:
>   “How strongly does this layer **transform** this token direction?”
> - **Dual role**: `W_ffn2_read`
>   The actual FFN output matrix has shape `(384, 1536)`. For probing, we use its transpose `(1536, 384)` and compute $\|W_\text{ffn2}^T x\|_2 / \|W_\text{ffn2}\|_F$. This is a **readout** probe:
>   “Which FFN hidden neurons are most sensitive to this token direction?”
>
> This mirrors ArrowSpace’s feature‑spectral Laplacian, which operates on the transposed feature matrix $X^\top$ to study relations in feature space instead of item space. We therefore treat `W_ffn2_read` as a **separate dual/readout axis**, not directly interchangeable with the primal roles in pooled summaries.

In [12]:
# ── Projection roles used in activation probing ──────────────────────────────
# Primal roles operate directly on token directions x ∈ ℝ^384 and measure
# how strongly each layer transforms / amplifies x.
#
# Dual role W_ffn2_read uses the transpose of the FFN output matrix:
#   W_ffn2:     (384, 1536) — maps FFN hidden → residual stream (primal output)
#   W_ffn2.T:   (1536, 384) — maps token direction → FFN hidden (dual/readout)
# and measures which FFN hidden neurons are most sensitive to a given x.
#
# This is analogous to ArrowSpace's feature‑spectral Laplacian, which operates
# on X^T to study feature‑space structure. We treat W_ffn2_read as a separate
# dual/readout axis rather than pooling it with the primal roles. [cite:25]

PRIMAL_ROLES = ["W_q", "W_k", "W_v", "W_o", "W_ffn1"]
ROLES_ATT    = PRIMAL_ROLES + ["W_ffn2_read"]

def activation_energy(W, x, role=""):
    """
    Scalar activation energy: ||W @ x||_2 / ||W||_F (Frobenius-normalised projection norm).

    Primal roles (W_q/k/v/o/ffn1): W @ x, x ∈ ℝ^384 → standard projection norm.

    Dual role (W_ffn2_read): W_ffn2.T @ x — measures FFN neuron activation
    sensitivity to token direction x. Analogous to ArrowSpace's feature-spectral
    Laplacian (X^T view): asks "which FFN neurons fire for this token direction?"
    NOT comparable to ffn1 on exactly the same semantic axis — treat as a separate
    readout/spectral dimension.

    NOTE on self-consistency bias (Probe B / X_pass):
      When x = model.encode(word), x has already passed through all W matrices.
      Projecting x back through e.g. W_q at layer 3 measures how strongly the
      weights recognise their own intermediate output — not how they transform
      an external signal. This inflates activation energies for semantically
      salient fields relative to Probe A (X_etok).

    Probe A (X_etok) is free of this bias: x is the raw pre-attention token
    direction, so activation_energy(W, x) is a clean measure of the weight
    geometry acting on an unprocessed input.
    """
    if role == "W_ffn2_read":
        W = W.T   # (384, 1536).T → (1536, 384)
    elif role == "W_ffn2":
        # Raw W_ffn2 is (384, 1536) — needs a 1536-dim input.
        # When called from ALL_ROLES with a 384-dim word_vec, skip gracefully.
        if x.shape[0] != W.shape[1]:
            return 0.0
    proj = W @ x
    return float(np.linalg.norm(proj) / (np.linalg.norm(W, "fro") + 1e-9))


N        = len(words)
n_layers = len(layers)
n_roles  = len(ROLES_ATT)
col_names = [f"L{i}_{r}" for i in range(n_layers) for r in ROLES_ATT]

# ── Probe A: activation matrix from raw E_tok lookup (X_etok) ──────────────
act_matrix_etok = np.zeros((N, n_layers * n_roles))
for n_idx, word_vec in enumerate(X_etok):
    col = 0
    for i in range(n_layers):
        for role in ROLES_ATT:
            W = weights[(i, "W_ffn2")] if role == "W_ffn2_read" else weights[(i, role)]
            act_matrix_etok[n_idx, col] = activation_energy(W, word_vec, role=role)
            col += 1

df_act_etok = pd.DataFrame(act_matrix_etok, columns=col_names)
df_act_etok["word"]  = words
df_act_etok["field"] = labels

# ── Probe B: activation matrix from full transformer pass (X_pass) ─────────
act_matrix = np.zeros((N, n_layers * n_roles))
for n_idx, word_vec in enumerate(X_pass):
    col = 0
    for i in range(n_layers):
        for role in ROLES_ATT:
            W = weights[(i, "W_ffn2")] if role == "W_ffn2_read" else weights[(i, role)]
            act_matrix[n_idx, col] = activation_energy(W, word_vec, role=role)
            col += 1

df_act = pd.DataFrame(act_matrix, columns=col_names)
df_act["word"]  = words
df_act["field"] = labels

# ── Summary comparison ─────────────────────────────────────────────────────────
print("Activation matrix (Probe A — E_tok lookup) shape:", act_matrix_etok.shape)
print("Activation matrix (Probe B — full pass) shape:", act_matrix.shape)

# Columns for primal and dual roles
PRIMAL_COLS = [c for c in col_names if any(r in c for r in PRIMAL_ROLES)]
DUAL_COLS   = [c for c in col_names if "W_ffn2_read" in c]

# Per-field mean energy across ALL PRIMAL subspaces for both strategies
mean_A_primal = (
    df_act_etok.groupby("field")[PRIMAL_COLS].mean().mean(axis=1)
    .rename("mean_energy_A_primal")   # Probe A = E_tok
)
mean_B_primal = (
    df_act.groupby("field")[PRIMAL_COLS].mean().mean(axis=1)
    .rename("mean_energy_B_primal")   # Probe B = full pass
)

# Optional: dual/readout only, as separate diagnostic
mean_A_dual = (
    df_act_etok.groupby("field")[DUAL_COLS].mean().mean(axis=1)
    .rename("mean_energy_A_dual")
)
mean_B_dual = (
    df_act.groupby("field")[DUAL_COLS].mean().mean(axis=1)
    .rename("mean_energy_B_dual")
)

df_compare = pd.concat([mean_A_primal, mean_B_primal, mean_A_dual, mean_B_dual], axis=1)
df_compare["delta_primal"]     = df_compare["mean_energy_B_primal"] - df_compare["mean_energy_A_primal"]
df_compare["delta_primal_%"]   = (df_compare["delta_primal"] / df_compare["mean_energy_A_primal"] * 100).round(1)
df_compare["delta_dual"]       = df_compare["mean_energy_B_dual"] - df_compare["mean_energy_A_dual"]
df_compare["delta_dual_%"]     = (df_compare["delta_dual"] / df_compare["mean_energy_A_dual"] * 100).round(1)

print("\nPer-field mean activation energy — Probe A vs B (primal roles only):")
print(df_compare.sort_values("delta_primal_%", ascending=False)[
    ["mean_energy_A_primal", "mean_energy_B_primal", "delta_primal", "delta_primal_%"]
].to_string())

print("\nPer-field mean activation energy — Probe A vs B (dual/readout role only):")
print(df_compare.sort_values("delta_dual_%", ascending=False)[
    ["mean_energy_A_dual", "mean_energy_B_dual", "delta_dual", "delta_dual_%"]
].to_string())

print("\nInterpretation (primal): large delta_primal_% → layer processing adds")
print("significant directional energy vs raw token geometry in the write/encode subspace.")

print("Interpretation (dual):   delta_dual_% tracks how much the FFN readout neurons")
print("increase or damp energy when measured via W_ffn2_read (feature/dual space).")

Activation matrix (Probe A — E_tok lookup) shape: (80, 36)
Activation matrix (Probe B — full pass) shape: (80, 36)

Per-field mean activation energy — Probe A vs B (primal roles only):
         mean_energy_A_primal  mean_energy_B_primal  delta_primal  delta_primal_%
field                                                                            
COLOUR               0.046952              0.052765      0.005813            12.4
FOOD                 0.046789              0.051971      0.005182            11.1
SCIENCE              0.047803              0.052135      0.004331             9.1
TOOL                 0.047820              0.051270      0.003449             7.2

Per-field mean activation energy — Probe A vs B (dual/readout role only):
         mean_energy_A_dual  mean_energy_B_dual  delta_dual  delta_dual_%
field                                                                    
FOOD               0.050305            0.058261    0.007956          15.8
TOOL               0.05060

### 3.1 — Probe A: Semantic Subspace Matrix Diagram


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# §3.1 Probe A  Semantic Subspace Matrix Diagram — raw E_tok lookup
# Answers: which (layer, weight-role) subspace encodes which semantic field
#          when probed with the PRE-ATTENTION token direction?
# Contrast with Probe B (X_pass) above to see how much the transformer
# reshapes the raw token geometry.
# ─────────────────────────────────────────────────────────────────────────────

ALL_ROLES   = ["W_q", "W_k", "W_v", "W_o", "W_ffn1", "W_ffn2_read"]
field_names = list(SEMANTIC_FIELDS.keys())
n_fields    = len(field_names)

ROLE_FAMILY = {
    "W_q": "attn", "W_k": "attn", "W_v": "attn", "W_o": "attn",
    "W_ffn1": "ffn_primal", "W_ffn2_read": "ffn_dual",
}

# ── 1. Build full activation matrix for Probe A (X_etok) ─────────────────────
col_names_full = [f"L{i}_{r}" for i in range(n_layers) for r in ALL_ROLES]
act_full_etok = np.zeros((N, n_layers * len(ALL_ROLES)))

for n_idx, word_vec in enumerate(X_etok):          # <── X_etok, not X_pass
    col = 0
    for i in range(n_layers):
        for role in ALL_ROLES:
            W = weights[(i, "W_ffn2")] if role == "W_ffn2_read" else weights[(i, role)]
            act_full_etok[n_idx, col] = activation_energy(W, word_vec, role=role)
            col += 1

df_full_etok = pd.DataFrame(act_full_etok, columns=col_names_full)
df_full_etok["word"]  = words
df_full_etok["field"] = labels

# ── 2. Compute per-(subspace, field) contrast score S ────────────────────────
subspace_keys = [(i, r) for i in range(n_layers) for r in ALL_ROLES]
n_subspaces   = len(subspace_keys)

S_matrix_etok = np.zeros((n_subspaces, n_fields))

for s_idx, (i, role) in enumerate(subspace_keys):
    col = f"L{i}_{role}"
    for f_idx, field in enumerate(field_names):
        mask_f  = labels == field
        mask_nf = ~mask_f
        mu_f    = df_full_etok.loc[mask_f,  col].mean()
        mu_nf   = df_full_etok.loc[mask_nf, col].mean()
        S_matrix_etok[s_idx, f_idx] = (mu_f - mu_nf) / (mu_f + mu_nf + 1e-9)

dominant_field_idx_etok = np.argmax(S_matrix_etok, axis=1)

# ── 3. Per-subspace top-3 contributing words (Probe A) ───────────────────────
def top3_words_for_subspace_etok(i, role, field):
    col  = f"L{i}_{role}"
    mask = labels == field
    sub  = df_full_etok.loc[mask, ["word", col]].nlargest(3, col)
    return ", ".join(sub["word"].tolist())

# ── 4. Build Plotly scatter (Probe A SSM) ────────────────────────────────────
field_colour_map = {f: FIELD_COLOURS[f] for f in field_names}
subspace_labels  = [f"L{i} · {r}" for (i, r) in subspace_keys]

DOT_MAX = 36
DOT_MIN = 4

fig_ssm_etok = go.Figure()

for f_idx, field in enumerate(field_names):
    xs, ys, sizes, hover_texts = [], [], [], []

    for s_idx, (i, role) in enumerate(subspace_keys):
        score       = S_matrix_etok[s_idx, f_idx]
        is_dominant = (dominant_field_idx_etok[s_idx] == f_idx)
        xs.append(f_idx)
        ys.append(s_idx)
        raw_size = DOT_MIN + (DOT_MAX - DOT_MIN) * abs(score)
        sizes.append(float(np.clip(raw_size, DOT_MIN, DOT_MAX)))
        top3 = top3_words_for_subspace_etok(i, role, field) if score > 0 else "—"
        hover_texts.append(
            f"<b>{field}</b> in {subspace_labels[s_idx]}<br>"
            f"Contrast S = {score:.3f}<br>"
            f"Dominant: {'✓' if is_dominant else '✗'}<br>"
            f"Role family: {ROLE_FAMILY[role]}<br>"
            f"Top words: {top3}"
        )

    dom_mask  = [dominant_field_idx_etok[s_idx] == f_idx for s_idx in range(n_subspaces)]
    ndom_mask = [not m for m in dom_mask]

    fig_ssm_etok.add_trace(go.Scatter(
        x=[f_idx for s_idx, m in enumerate(dom_mask) if m],
        y=[s_idx for s_idx, m in enumerate(dom_mask) if m],
        mode="markers",
        marker=dict(
            color=field_colour_map[field],
            size=[sizes[s_idx] for s_idx, m in enumerate(dom_mask) if m],
            symbol="circle", line=dict(width=0), opacity=1.0,
        ),
        text=[hover_texts[s_idx] for s_idx, m in enumerate(dom_mask) if m],
        hovertemplate="%{text}<extra></extra>",
        name=field, legendgroup=field, showlegend=True,
    ))

    if any(ndom_mask):
        fig_ssm_etok.add_trace(go.Scatter(
            x=[f_idx for s_idx, m in enumerate(ndom_mask) if m],
            y=[s_idx for s_idx, m in enumerate(ndom_mask) if m],
            mode="markers",
            marker=dict(
                color="rgba(0,0,0,0)",
                size=[max(DOT_MIN, sizes[s_idx] * 0.55)
                      for s_idx, m in enumerate(ndom_mask) if m],
                symbol="circle",
                line=dict(width=1.2, color=field_colour_map[field]),
                opacity=0.35,
            ),
            text=[hover_texts[s_idx] for s_idx, m in enumerate(ndom_mask) if m],
            hovertemplate="%{text}<extra></extra>",
            name=field, legendgroup=field, showlegend=False,
        ))

# ── 5. Separator lines and layer annotations ─────────────────────────────────
for i in range(1, n_layers):
    sep_y = i * len(ALL_ROLES) - 0.5
    fig_ssm_etok.add_shape(
        type="line",
        x0=-0.5, x1=n_fields - 0.5,
        y0=sep_y, y1=sep_y,
        line=dict(color="rgba(120,120,120,0.35)", width=1, dash="dot"),
    )

for i in range(n_layers):
    mid_y = i * len(ALL_ROLES) + (len(ALL_ROLES) - 1) / 2
    fig_ssm_etok.add_annotation(
        x=n_fields - 0.1, y=mid_y,
        text=f"<b>Layer {i}</b>",
        showarrow=False, xanchor="left",
        font=dict(size=10, color="#666", family="monospace"),
        xref="x", yref="y",
    )

# ── 6. Layout ─────────────────────────────────────────────────────────────────
fig_ssm_etok.update_layout(
    title=dict(
        text=(
            "Semantic Subspace Matrix — Probe A (raw E_tok) · "
            "which (layer × weight-role) encodes which field?<br>"
            "<sup>Filled dot = dominant · Hollow = secondary · "
            "Size ∝ contrast S · Compare with Probe B above to see transformer reshaping</sup>"
        ),
        font=dict(size=13, family="monospace"),
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(n_fields)),
        ticktext=[f"<b>{f}</b>" for f in field_names],
        tickfont=dict(size=10, family="monospace"),
        title="Semantic Field",
        showgrid=False, zeroline=False, side="top",
        range=[-0.6, n_fields - 0.4],
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(n_subspaces)),
        ticktext=[f"<span style='font-family:monospace;font-size:10px'>{lbl}</span>"
                  for lbl in subspace_labels],
        tickfont=dict(size=10, family="monospace"),
        title="Weight-role subspace",
        autorange="reversed",
        showgrid=False, zeroline=False,
    ),
    plot_bgcolor="#f9f8f5", paper_bgcolor="#ffffff",
    height=820, width=1050,
    margin=dict(l=90, r=120, t=110, b=30),
    legend=dict(
        title="Semantic Field", orientation="v",
        x=1.01, y=1.0,
        font=dict(size=10, family="monospace"),
        itemsizing="constant", tracegroupgap=2,
    ),
    font=dict(family="monospace"),
    hoverlabel=dict(font_family="monospace"),
)

fig_ssm_etok.write_image(OUTPUT_DIR / "fig_03a_semantic_subspace_matrix_etok.png", scale=2)
fig_ssm_etok.show()
print("Saved fig_03a_semantic_subspace_matrix_etok.png")

# ── 7. Summary table ──────────────────────────────────────────────────────────
df_ssm_etok = pd.DataFrame({
    "Subspace":       subspace_labels,
    "Dominant Field": [field_names[i] for i in dominant_field_idx_etok],
    "Contrast S":     [round(S_matrix_etok[s, dominant_field_idx_etok[s]], 4)
                       for s in range(n_subspaces)],
    "Top Words":      [top3_words_for_subspace_etok(*subspace_keys[s],
                           field_names[dominant_field_idx_etok[s]])
                       for s in range(n_subspaces)],
})
df_ssm_etok.to_csv(OUTPUT_DIR / "semantic_subspace_ownership_etok.csv", index=False)
print("\n=== Dominant field per subspace (Probe A — E_tok) ===")
print(df_ssm_etok.to_string(index=False))

Saved fig_03a_semantic_subspace_matrix_etok.png

=== Dominant field per subspace (Probe A — E_tok) ===
        Subspace Dominant Field  Contrast S                 Top Words
        L0 · W_q           TOOL      0.0168       wheel, forge, knife
        L0 · W_k        SCIENCE      0.0192     protein, energy, gene
        L0 · W_v        SCIENCE      0.0228   photon, energy, protein
        L0 · W_o           TOOL      0.0103          hook, hook, bolt
     L0 · W_ffn1        SCIENCE      0.0121  circuit, enzyme, neutron
L0 · W_ffn2_read        SCIENCE      0.0111    enzyme, virus, neutron
        L1 · W_q           TOOL      0.0125         wheel, forge, saw
        L1 · W_k           FOOD      0.0104       beer, pizza, coffee
        L1 · W_v           TOOL      0.0204         hook, hook, forge
        L1 · W_o        SCIENCE      0.0020         virus, mass, gene
     L1 · W_ffn1           TOOL      0.0091          saw, forge, bolt
L1 · W_ffn2_read        SCIENCE      0.0069    virus, enz

### 3.1 — Probe B: Semantic Subspace Matrix Diagram

In [14]:
# ── §3.1 Probe B  Semantic Subspace Matrix Diagram — full transformer pass ──
# Mirrors §3.1 Probe A exactly, but uses X_pass (model.encode) instead of X_etok.
# The comparison between the two SSMs reveals transformer reshaping per subspace.
# ─────────────────────────────────────────────────────────────────────────────

print(FIELD_COLOURS)  # sanity-check colours are in scope

ALL_ROLES   = ["W_q", "W_k", "W_v", "W_o", "W_ffn1", "W_ffn2_read"]
field_names = list(SEMANTIC_FIELDS.keys())
n_fields    = len(field_names)

ROLE_FAMILY = {
    "W_q": "attn", "W_k": "attn", "W_v": "attn", "W_o": "attn",
    "W_ffn1": "ffn_primal", "W_ffn2_read": "ffn_dual",
}

# ── 1. Build full activation matrix for Probe B (X_pass) ────────────────────
col_names_full = [f"L{i}_{r}" for i in range(n_layers) for r in ALL_ROLES]
act_full_pass = np.zeros((N, n_layers * len(ALL_ROLES)))

for n_idx, word_vec in enumerate(X_pass):          # <── X_pass, not X_etok
    col = 0
    for i in range(n_layers):
        for role in ALL_ROLES:
            W = weights[(i, "W_ffn2")] if role == "W_ffn2_read" else weights[(i, role)]
            act_full_pass[n_idx, col] = activation_energy(W, word_vec, role=role)
            col += 1

df_full_pass = pd.DataFrame(act_full_pass, columns=col_names_full)
df_full_pass["word"]  = words
df_full_pass["field"] = labels

# ── 2. Compute per-(subspace, field) contrast score S ────────────────────────
subspace_keys  = [(i, r) for i in range(n_layers) for r in ALL_ROLES]
n_subspaces    = len(subspace_keys)
subspace_labels = [f"L{i} · {r}" for (i, r) in subspace_keys]

S_matrix_pass = np.zeros((n_subspaces, n_fields))

for s_idx, (i, role) in enumerate(subspace_keys):
    col = f"L{i}_{role}"
    for f_idx, field in enumerate(field_names):
        mask_f  = labels == field
        mask_nf = ~mask_f
        mu_f    = df_full_pass.loc[mask_f,  col].mean()
        mu_nf   = df_full_pass.loc[mask_nf, col].mean()
        S_matrix_pass[s_idx, f_idx] = (mu_f - mu_nf) / (mu_f + mu_nf + 1e-9)

dominant_field_idx_pass = np.argmax(S_matrix_pass, axis=1)

# ── 3. Per-subspace top-3 contributing words (Probe B) ───────────────────────
def top3_words_for_subspace_pass(i, role, field):
    col  = f"L{i}_{role}"
    mask = labels == field
    sub  = df_full_pass.loc[mask, ["word", col]].nlargest(3, col)
    return ", ".join(sub["word"].tolist())

# ── 4. Build Plotly scatter (Probe B SSM) ────────────────────────────────────
field_colour_map = {f: FIELD_COLOURS[f] for f in field_names}
DOT_MAX = 36
DOT_MIN = 4

fig_ssm_pass = go.Figure()

for f_idx, field in enumerate(field_names):
    xs, ys, sizes, hover_texts = [], [], [], []

    for s_idx, (i, role) in enumerate(subspace_keys):
        score       = S_matrix_pass[s_idx, f_idx]
        is_dominant = (dominant_field_idx_pass[s_idx] == f_idx)
        xs.append(f_idx)
        ys.append(s_idx)
        raw_size = DOT_MIN + (DOT_MAX - DOT_MIN) * abs(score)
        sizes.append(float(np.clip(raw_size, DOT_MIN, DOT_MAX)))
        top3 = top3_words_for_subspace_pass(i, role, field) if score > 0 else "—"
        hover_texts.append(
            f"<b>{field}</b> in {subspace_labels[s_idx]}<br>"
            f"Contrast S = {score:.3f}<br>"
            f"Dominant: {'✓' if is_dominant else '✗'}<br>"
            f"Role family: {ROLE_FAMILY[role]}<br>"   # ← FIX: was missing
            f"Top words: {top3}"
        )

    dom_mask  = [dominant_field_idx_pass[s_idx] == f_idx for s_idx in range(n_subspaces)]
    ndom_mask = [not m for m in dom_mask]

    fig_ssm_pass.add_trace(go.Scatter(
        x=[f_idx for s_idx, m in enumerate(dom_mask) if m],
        y=[s_idx for s_idx, m in enumerate(dom_mask) if m],
        mode="markers",
        marker=dict(
            color=field_colour_map[field],
            size=[sizes[s_idx] for s_idx, m in enumerate(dom_mask) if m],
            symbol="circle", line=dict(width=0), opacity=1.0,
        ),
        text=[hover_texts[s_idx] for s_idx, m in enumerate(dom_mask) if m],
        hovertemplate="%{text}<extra></extra>",
        name=field, legendgroup=field, showlegend=True,
    ))

    if any(ndom_mask):
        fig_ssm_pass.add_trace(go.Scatter(
            x=[f_idx for s_idx, m in enumerate(ndom_mask) if m],
            y=[s_idx for s_idx, m in enumerate(ndom_mask) if m],
            mode="markers",
            marker=dict(
                color="rgba(0,0,0,0)",
                size=[max(DOT_MIN, sizes[s_idx] * 0.55)
                      for s_idx, m in enumerate(ndom_mask) if m],
                symbol="circle",
                line=dict(width=1.2, color=field_colour_map[field]),
                opacity=0.35,
            ),
            text=[hover_texts[s_idx] for s_idx, m in enumerate(ndom_mask) if m],
            hovertemplate="%{text}<extra></extra>",
            name=field, legendgroup=field, showlegend=False,
        ))

# ── 5. Separator lines and layer annotations ─────────────────────────────────
for i in range(1, n_layers):
    sep_y = i * len(ALL_ROLES) - 0.5
    fig_ssm_pass.add_shape(
        type="line",
        x0=-0.5, x1=n_fields - 0.5,
        y0=sep_y, y1=sep_y,
        line=dict(color="rgba(120,120,120,0.35)", width=1, dash="dot"),
    )

for i in range(n_layers):
    mid_y = i * len(ALL_ROLES) + (len(ALL_ROLES) - 1) / 2
    fig_ssm_pass.add_annotation(
        x=n_fields - 0.1, y=mid_y,
        text=f"<b>Layer {i}</b>",
        showarrow=False, xanchor="left",
        font=dict(size=10, color="#666", family="monospace"),
        xref="x", yref="y",
    )

# ── 6. Layout ─────────────────────────────────────────────────────────────────
fig_ssm_pass.update_layout(
    title=dict(
        text=(
            "Semantic Subspace Matrix — Probe B (full transformer pass) · "
            "which (layer × weight-role) encodes which field?<br>"
            "<sup>Filled dot = dominant · Hollow = secondary · "
            "Size ∝ contrast S · Compare with Probe A (E_tok) to see transformer reshaping</sup>"
        ),
        font=dict(size=13, family="monospace"),
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(n_fields)),
        ticktext=[f"<b>{f}</b>" for f in field_names],
        tickfont=dict(size=10, family="monospace"),
        title="Semantic Field",
        showgrid=False, zeroline=False, side="top",
        range=[-0.6, n_fields - 0.4],
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(n_subspaces)),
        ticktext=[f"<span style='font-family:monospace;font-size:10px'>{lbl}</span>"
                  for lbl in subspace_labels],
        tickfont=dict(size=10, family="monospace"),
        title="Weight-role subspace",
        autorange="reversed",
        showgrid=False, zeroline=False,
    ),
    plot_bgcolor="#f9f8f5", paper_bgcolor="#ffffff",
    height=820, width=1050,
    margin=dict(l=90, r=120, t=110, b=30),
    legend=dict(
        title="Semantic Field", orientation="v",
        x=1.01, y=1.0,
        font=dict(size=10, family="monospace"),
        itemsizing="constant", tracegroupgap=2,
    ),
    font=dict(family="monospace"),
    hoverlabel=dict(font_family="monospace"),
)

fig_ssm_pass.write_image(OUTPUT_DIR / "fig_03b_semantic_subspace_matrix_pass.png", scale=2)
fig_ssm_pass.show()
print("Saved fig_03b_semantic_subspace_matrix_pass.png")

# ── 7. Summary table ──────────────────────────────────────────────────────────
df_ssm_pass = pd.DataFrame({
    "Subspace":       subspace_labels,
    "Dominant Field": [field_names[i] for i in dominant_field_idx_pass],
    "Contrast S":     [round(S_matrix_pass[s, dominant_field_idx_pass[s]], 4)
                       for s in range(n_subspaces)],
    "Top Words":      [top3_words_for_subspace_pass(*subspace_keys[s],
                           field_names[dominant_field_idx_pass[s]])
                       for s in range(n_subspaces)],
})
df_ssm_pass.to_csv(OUTPUT_DIR / "semantic_subspace_ownership_pass.csv", index=False)
print("\n=== Dominant field per subspace (Probe B — full pass) ===")
print(df_ssm_pass.to_string(index=False))

{'ANIMAL': '#e6194b', 'FOOD': '#f58231', 'EMOTION': '#ffe119', 'SCIENCE': '#3cb44b', 'PLACE': '#42d4f4', 'TOOL': '#4363d8', 'MUSIC': '#911eb4', 'COLOUR': '#f032e6', 'ACTION': '#a9a9a9', 'ABSTRACT': '#9A6324'}


Saved fig_03b_semantic_subspace_matrix_pass.png

=== Dominant field per subspace (Probe B — full pass) ===
        Subspace Dominant Field  Contrast S                Top Words
        L0 · W_q         COLOUR      0.0147        gold, brown, grey
        L0 · W_k           FOOD      0.0158       salad, cake, cream
        L0 · W_v        SCIENCE      0.0130      neutron, atom, atom
        L0 · W_o           TOOL      0.0052        hook, hook, forge
     L0 · W_ffn1         COLOUR      0.0498       blue, black, brown
L0 · W_ffn2_read           FOOD      0.0287       cake, pasta, cream
        L1 · W_q         COLOUR      0.0146        gold, white, blue
        L1 · W_k           FOOD      0.0113      cheese, salad, cake
        L1 · W_v         COLOUR      0.0081    purple, indigo, black
        L1 · W_o        SCIENCE      0.0046    entropy, mass, photon
     L1 · W_ffn1         COLOUR      0.0114      silver, gold, white
L1 · W_ffn2_read           FOOD      0.0179      pasta, cheese, c

### 3.1 — DUAL SPACE COMPARISON — ΔS heatmap: Probe B minus Probe A

In [15]:
# ── §3.1 DUAL SPACE COMPARISON — ΔS heatmap: Probe B minus Probe A ──────────
#
# ΔS(subspace, field) = S_pass(s,f) − S_etok(s,f)
#
# Interpretation:
#   ΔS > 0 → transformer AMPLIFIES this field's signal in this subspace
#             (full-pass embedding is MORE separable than raw token here)
#   ΔS < 0 → transformer SUPPRESSES this field's signal in this subspace
#             (raw token direction was stronger; layers wash it out)
#   ΔS ≈ 0 → layers add no new geometric structure for this field/subspace pair
#
# This directly operationalises the theoretical claim in the markdown above:
# "Divergence between Probe A and Probe B indicates the layers do non-trivial
# re-encoding." High |ΔS| subspaces are prime candidates for ArrowSpace's
# R_spec (spectral component) to capture at inference time.
# ─────────────────────────────────────────────────────────────────────────────

delta_S = S_matrix_pass - S_matrix_etok   # (n_subspaces, n_fields)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Heatmap ──────────────────────────────────────────────────────────────────
# Rows = subspaces (layer × role), Cols = semantic fields
# Colour = ΔS, symmetric diverging scale centred at 0

abs_max = float(np.abs(delta_S).max()) * 1.05   # symmetric colour range

fig_delta = go.Figure(go.Heatmap(
    z=delta_S,
    x=field_names,
    y=subspace_labels,
    colorscale="RdBu",
    zmid=0,
    zmin=-abs_max,
    zmax=abs_max,
    colorbar=dict(
        title="ΔS (B − A)",
        tickfont=dict(family="monospace", size=10),
        len=0.9,
    ),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Field: <b>%{x}</b><br>"
        "ΔS = %{z:.4f}<br>"
        "<extra></extra>"
    ),
    xgap=1,
    ygap=1,
))

# Layer separator lines
for i in range(1, n_layers):
    sep_y = i * len(ALL_ROLES) - 0.5
    fig_delta.add_shape(
        type="line",
        x0=-0.5, x1=n_fields - 0.5,
        y0=sep_y, y1=sep_y,
        line=dict(color="rgba(80,80,80,0.6)", width=1.5, dash="solid"),
    )

# Layer annotations (right side)
for i in range(n_layers):
    mid_y = i * len(ALL_ROLES) + (len(ALL_ROLES) - 1) / 2
    fig_delta.add_annotation(
        x=n_fields - 0.5 + 0.15, y=mid_y,
        text=f"<b>L{i}</b>",
        showarrow=False, xanchor="left",
        font=dict(size=10, color="#555", family="monospace"),
        xref="x", yref="y",
    )

fig_delta.update_layout(
    title=dict(
        text=(
            "§3.1 Dual Space ΔS — Probe B (full pass) minus Probe A (E_tok)<br>"
            "<sup>Red = transformer amplifies field signal · Blue = transformer suppresses · "
            "White = no reshaping · Layer separators shown</sup>"
        ),
        font=dict(size=13, family="monospace"),
    ),
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(n_fields)),
        ticktext=[f"<b>{f}</b>" for f in field_names],
        tickfont=dict(size=11, family="monospace"),
        title="Semantic Field",
        side="top",
        showgrid=False,
    ),
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(n_subspaces)),
        ticktext=[f"<span style='font-family:monospace;font-size:9px'>{lbl}</span>"
                  for lbl in subspace_labels],
        tickfont=dict(size=9, family="monospace"),
        title="Weight-role subspace (layer × role)",
        autorange="reversed",
        showgrid=False,
    ),
    plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
    height=920, width=820,
    margin=dict(l=130, r=100, t=120, b=30),
    font=dict(family="monospace"),
    hoverlabel=dict(font_family="monospace"),
)

fig_delta.write_image(OUTPUT_DIR / "fig_03c_delta_S_dual_space.png", scale=2)
fig_delta.show()
print("Saved fig_03c_delta_S_dual_space.png")

# ── Ranked ΔS summary ─────────────────────────────────────────────────────────
# Flatten and find the highest-|ΔS| (layer, role, field) triplets
rows = []
for s_idx, (i, role) in enumerate(subspace_keys):
    for f_idx, field in enumerate(field_names):
        rows.append({
            "Subspace":    subspace_labels[s_idx],
            "Layer":       i,
            "Role":        role,
            "Role family": ROLE_FAMILY[role],
            "Field":       field,
            "S_A":         round(S_matrix_etok[s_idx, f_idx], 4),
            "S_B":         round(S_matrix_pass[s_idx, f_idx], 4),
            "delta_S":     round(delta_S[s_idx, f_idx], 4),
            "abs_delta_S": round(abs(delta_S[s_idx, f_idx]), 4),
        })

df_delta = pd.DataFrame(rows).sort_values("abs_delta_S", ascending=False).reset_index(drop=True)
df_delta.to_csv(OUTPUT_DIR / "delta_S_dual_space.csv", index=False)

print("\n=== Top-20 subspace × field pairs by |ΔS| (transformer reshaping magnitude) ===")
print(df_delta.head(20)[
    ["Subspace", "Field", "Role family", "S_A", "S_B", "delta_S"]
].to_string(index=False))

# ── Per-role-family mean |ΔS| ─────────────────────────────────────────────────
print("\n=== Mean |ΔS| by role family (primal vs dual) ===")
print(df_delta.groupby("Role family")["abs_delta_S"].mean().round(4).to_string())

print("\n=== Mean |ΔS| by layer ===")
print(df_delta.groupby("Layer")["abs_delta_S"].mean().round(4).to_string())

Saved fig_03c_delta_S_dual_space.png

=== Top-20 subspace × field pairs by |ΔS| (transformer reshaping magnitude) ===
        Subspace   Field Role family     S_A     S_B  delta_S
        L2 · W_v  COLOUR        attn -0.0399  0.0203   0.0602
        L2 · W_v    TOOL        attn  0.0232 -0.0303  -0.0535
     L0 · W_ffn1  COLOUR  ffn_primal  0.0042  0.0498   0.0456
        L0 · W_v  COLOUR        attn -0.0318  0.0110   0.0427
        L4 · W_v    TOOL        attn  0.0098 -0.0318  -0.0416
        L4 · W_k    TOOL        attn  0.0189 -0.0222  -0.0411
        L5 · W_v  COLOUR        attn  0.0077 -0.0318  -0.0394
        L4 · W_q  COLOUR        attn  0.0030  0.0411   0.0381
     L0 · W_ffn1    TOOL  ffn_primal  0.0022 -0.0359  -0.0380
        L4 · W_q    TOOL        attn  0.0205 -0.0161  -0.0365
        L1 · W_v    TOOL        attn  0.0204 -0.0161  -0.0365
        L5 · W_v    TOOL        attn -0.0031  0.0334   0.0364
        L3 · W_v  COLOUR        attn -0.0185  0.0173   0.0359
L0 · W_ffn2_re

## §4 — Signed Laplacian Extension: $L_\pm$ as a Real-Valued ±Amplitude Density

### Theoretical Framing

The FFN output weight matrix $W_\text{ffn2}$ has shape $(384, 1536)$ — it maps from the FFN hidden space back into the residual stream. Its **Gram matrix**

$$
G_\pm = W_\text{ffn2}\, W_\text{ffn2}^\top \in \mathbb{R}^{384 \times 384}
$$

is a symmetric matrix in token space. Each off-diagonal entry $G_{ij}$ is the **dot product** of the two residual-stream directions written by FFN neurons $i$ and $j$: a positive value encodes **constructive interference** (both neurons push the residual stream in the same direction), while a negative value encodes **destructive interference** (they write in opposing directions). This signed coupling structure is precisely the ±amplitude interaction absent from unsigned activation norms.

### The Signed Normalised Laplacian

The **signed normalised Laplacian** is defined as

$$
L_\pm = D^{-1/2}\, G_\pm\, D^{-1/2}
$$

where the degree matrix $D$ is computed from **absolute row sums**,

$$
D_{ii} = \sum_j \lvert G_{ij} \rvert
$$

rather than from positive entries only. This is the critical departure from the standard graph Laplacian, which discards sign. Using absolute row sums ensures $D$ is well-defined and the normalisation is stable, while the off-diagonal sign of $G_\pm$ is fully preserved in $L_\pm$. The resulting matrix is real and symmetric, admitting a real eigendecomposition $L_\pm = \Phi\,\Lambda\,\Phi^\top$ via the standard symmetric eigensolver — no complex arithmetic is required at any step.

### Negative Eigenvalues as Anti-Phase Modes

In a standard (unsigned) graph Laplacian, all eigenvalues are non-negative by construction, because the matrix is positive semi-definite. $L_\pm$ carries **no such guarantee**: its eigenvalues $\lambda_k \in \mathbb{R}$ can be negative, and those negative eigenvalues are physically meaningful rather than numerical artefacts.

A **positive eigenvalue** corresponds to a mode in which neighbouring token directions in the Gram graph are in phase — the FFN reinforces their shared representation. This is the vibrational analogue of a **normal mode** with a restoring force: the low-entropy oscillating state of the string.

A **negative eigenvalue** corresponds to a mode in which neighbours are systematically anti-correlated — the FFN writes them in opposing residual-stream directions. This is the analogue of **anti-resonance** in Rayleigh's theory of sound, and maps directly onto Aaronson's negative amplitudes: probability-like weights that are real-valued but can destructively interfere, producing the characteristically quantum outcome of cancellation without complex phases.

### Signed Rayleigh Quotient as ±Amplitude Score

Projecting a token embedding $x \in \mathbb{R}^{384}$ onto $L_\pm$ via the **signed Rayleigh quotient**

$$
R_\pm(x) = \frac{x^\top L_\pm\, x}{x^\top x}
$$

yields a single real scalar per word per layer. When $R_\pm(x) > 0$, the token direction is **resonant** with the FFN's constructive modes at that layer — it sits in a semantic basin of positive amplitude. When $R_\pm(x) < 0$, the token is **anti-resonant**: it projects predominantly onto destructive-interference eigenmodes, the vibrational equivalent of a node on a standing wave.

This is the minimal structure needed to represent a **real-valued density matrix without complex numbers or Hermitian algebra**. The signed Gram matrix $G_\pm$ plays the role of the density operator; its normalised form $L_\pm$ is the graph-theoretic counterpart; and $R_\pm(x)$ is the expectation value of the observable $L_\pm$ in the state $x$. All three quantities live in $\mathbb{R}$, the eigenvectors of $L_\pm$ form an orthonormal real basis, and the full interference structure — constructive and destructive — is encoded in the sign of off-diagonal entries rather than in complex phases.

| Quantum / Hermitian concept | $L_\pm$ real-valued counterpart |
|---|---|
| Density matrix $\rho$ (Hermitian, complex) | Signed Gram $G_\pm = W_\text{ffn2} W_\text{ffn2}^\top$ (real, symmetric) |
| Complex amplitude $\alpha_k \in \mathbb{C}$ | Signed eigenvalue $\lambda_k \in \mathbb{R}$ |
| Destructive interference via phase $e^{i\pi}$ | Negative off-diagonal entry $G_{ij} < 0$ |
| Expectation value $\langle \psi \vert \hat{O} \vert \psi \rangle$ | Signed Rayleigh quotient $R_\pm(x) = x^\top L_\pm x / x^\top x$ |
| Positive semi-definite (PSD) constraint | Relaxed: $L_\pm$ is not PSD; negative $\lambda_k$ are admissible |



In [21]:
from scipy.linalg import eigh

def signed_normalised_laplacian(W_ffn2):
    G   = W_ffn2 @ W_ffn2.T                          # (384,384) signed Gram
    d   = np.abs(G).sum(axis=1)                       # absolute degree
    inv = np.where(d > 1e-12, d**-0.5, 0.0)
    D   = np.diag(inv)
    return G, D @ G @ D                               # raw Gram + L±

G_signed, L_pm, eigenvalues, eigenvectors = {}, {}, {}, {}
for i in range(n_layers):
    G_signed[i], L_pm[i] = signed_normalised_laplacian(weights[(i, "W_ffn2")])
    eigenvalues[i], eigenvectors[i] = eigh(L_pm[i])  # real spectrum, can be negative

def signed_rayleigh(L, x):
    return float(x @ L @ x) / (float(x @ x) + 1e-12)

signed_rq = np.zeros((N, n_layers))
for n_idx, word_vec in enumerate(X_etok):            # Probe A — no circularity
    for i in range(n_layers):
        signed_rq[n_idx, i] = signed_rayleigh(L_pm[i], word_vec)

df_srq = pd.DataFrame(signed_rq, columns=[f"L{i}_srq" for i in range(n_layers)])
df_srq["word"]  = words
df_srq["field"] = labels
df_srq.to_csv(OUTPUT_DIR / "signed_rayleigh_per_word_layer.csv", index=False)

for i in range(n_layers):
    np.save(OUTPUT_DIR / f"L_pm_eigenvalues_layer{i}.npy", eigenvalues[i])

neg_counts = {i: int((eigenvalues[i] < 0).sum()) for i in range(n_layers)}
print("Negative eigenvalue counts per layer:", neg_counts)
print("\nMean signed Rayleigh per field per layer:")
print(df_srq.groupby("field")[[f"L{i}_srq" for i in range(n_layers)]].mean().round(4).to_string())

Negative eigenvalue counts per layer: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0}

Mean signed Rayleigh per field per layer:
         L0_srq  L1_srq  L2_srq  L3_srq  L4_srq  L5_srq
field                                                  
COLOUR   0.0834  0.0594  0.0556  0.0636  0.0803  0.0764
FOOD     0.0892  0.0592  0.0569  0.0627  0.0777  0.0772
SCIENCE  0.0897  0.0613  0.0592  0.0657  0.0795  0.0757
TOOL     0.0863  0.0618  0.0602  0.0662  0.0787  0.0751


In [22]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=["L± eigenspectrum per layer (diverging)",
                    "Mean signed Rayleigh per field × layer"])

# Left: eigenvalue heatmap
eig_matrix = np.stack([eigenvalues[i] for i in range(n_layers)])  # (6, 384)
fig.add_trace(go.Heatmap(z=eig_matrix, colorscale="RdBu", zmid=0,
    x=list(range(384)), y=[f"L{i}" for i in range(n_layers)],
    colorbar=dict(x=0.45, title="λ")), row=1, col=1)

# Right: signed Rayleigh per field per layer
srq_means = df_srq.groupby("field")[[f"L{i}_srq" for i in range(n_layers)]].mean()
for field in srq_means.index:
    fig.add_trace(go.Scatter(
        x=list(range(n_layers)), y=srq_means.loc[field].values,
        mode="lines+markers", name=field,
        line=dict(color=FIELD_COLOURS.get(field, "#888"))), row=1, col=2)

fig.add_hline(y=0, line_dash="dot", line_color="grey", row=1, col=2)
fig.update_layout(title="Signed Laplacian L± — anti-phase modes and token resonance",
    height=500, width=1100, template="plotly_white")
fig.write_image(OUTPUT_DIR / "fig_04_signed_laplacian.png", scale=2)
fig.show()